# ELO Rating applied to International Football Sides

**Goal:** turn the cleaned match data from `international_match_data.ipynb`
into a single number per team, per point in time, that captures relative
team strength — an Elo rating. This rating is the core engineered feature
that will feed the Random Forest match-outcome model.

Standard chess Elo assumes a fixed-margin win/loss between equally-weighted
games. Football needs three adjustments to be a fair measure of team
strength:
- **Home advantage** — a home win is less surprising than an away win, so
  the model needs an offset that's removed for matches played at a neutral
  venue.
- **Match importance** — a World Cup result should move a rating more than
  a friendly. A per-tournament K-factor controls this.
- **Margin of victory** — a 4-0 win is stronger evidence than a 1-0 win, so
  the rating update scales with goal difference rather than being purely
  win/loss.

The functions developed here are promoted into
[`src/ml_world_cup_predictor/elo_creation.py`](../src/ml_world_cup_predictor/elo_creation.py)
and [`config.py`](../src/ml_world_cup_predictor/config.py) once validated.


In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import datetime as dt
from pathlib import Path


from ml_world_cup_predictor.data_loading import load_raw_data

DATA_DIRECTORY = Path("../data/international-football-results")


frames = load_raw_data(DATA_DIRECTORY)

results = frames['results']
shootouts = frames['shootouts']
goalscorers = frames['goalscorers']
former_names = frames['former_names']



c:\Users\eoinm\Documents\ml_world_cup_predictor\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Re-applying the cleaning steps from `international_match_data.ipynb`

The same data-quality fixes developed in the data exploration notebook are
repeated here against the shared `load_data` loader: appending the missing
Sierra Leone vs Liberia match, reconciling shootout winners, deriving the
`result`/`goal_diff` columns, and normalising 2026 World Cup squad names
against `former_names`.

In [2]:
missing_match = {
    'date':dt.datetime(2024,10,27),
    'home_team':'Sierra Leone',
    'away_team':'Liberia',
    'home_score':1,
    'away_score':2,
    'tournament':'African Cup of Nations qualification',
    'city':'Monrovia',
    'country':'Liberia',
    'neutral': False
}

row = pd.DataFrame(missing_match,index = [0])

results_new = pd.concat([results,row],ignore_index=True)

results_new = results_new.sort_values(by = 'date').reset_index(drop=True)


In [ ]:
not_played = results_new[results_new['home_score'].isna()].copy()
played = results_new[~results_new['home_score'].isna()].copy()

played['shootout_id'] = (played['date'].astype(str) + '_' + played['home_team'] + '_' + played['away_team'])
shootouts['shootout_id'] = (shootouts['date'].astype(str) + '_' + shootouts['home_team'] + '_' + shootouts['away_team'])

shootout_dict = pd.Series(shootouts['winner'].values, index = shootouts['shootout_id']).to_dict()

played['shootout_winner'] = played['shootout_id'].map(shootout_dict)

played['year'] = played['date'].dt.year
played_long = played.melt(
    id_vars=['date','year','tournament'],
    value_vars=['home_team','away_team'],
    value_name='team'
)


wc_matches = played_long[(played_long['tournament'] == 'FIFA World Cup') & (played_long['year'] == 2026)]
wc_teams = wc_matches['team'].unique()
wc_former_names = former_names[former_names['current'].isin(wc_teams)]

name_map = dict(zip(former_names['former'], former_names['current']))

played['home_team'] = played['home_team'].replace(name_map)
played['away_team'] = played['away_team'].replace(name_map)

played_long['team'] = played_long['team'].replace(name_map)

match_conditions = [
    played['home_score'] > played['away_score'],
    played['home_score'] < played['away_score'],
    played['shootout_winner'] == played['home_team'],
    played['shootout_winner'] == played['away_team']
]  

choices = ['W','L','W','L']
played['result'] = np.select(match_conditions,choices,default='D')
played['goal_diff'] = np.abs(played['home_score'] - played['away_score'])

## The Elo formulas

`elo_expected_outcome` is the standard logistic Elo win-probability formula,
extended with an `omega` term that shifts the effective rating gap in the
home team's favour (set to 0 for neutral-venue matches, applied otherwise).

`elo_rank_update` is the rating update step: the team's new rating moves
toward the actual result, scaled by `k0` (importance of the match/tournament)
and `(1 + goal_difference)` (margin of victory). `generate_team_list` and
`result_convertion` are small helpers — the former restricts the rating pool
to teams with a minimum number of games so a handful of one-off results
can't swing the rankings, the latter converts a `W`/`L`/`D` label to the
(1, 0)/(0, 1)/(0.5, 0.5) outcome pair the update step expects.

In [ ]:
def elo_expected_outcome(home_rating,away_rating,c =10,d=400,omega = 50):

    exponent = (away_rating - home_rating - omega)/d

    return 1/(1+c**(exponent))

def elo_rank_update(previous_rating,goal_difference,actual_outcome,expected_outcome,k0 =50):

    importance_factor = k0 * (1+goal_difference)
    
    return previous_rating + importance_factor * (actual_outcome - expected_outcome)


def generate_team_list(df,minimum_games):

    df =df.copy()
    
    df = df.melt(
    id_vars=['date','year','tournament'],
    value_vars=['home_team','away_team'],
    value_name='team'
)
    count = df['team'].value_counts()
    count = count[count >= minimum_games]

    return sorted(list(count.index)) 


def result_convertion(result):
        if result == 'W':
            return 1,0
        elif result == 'L':
            return 0,1
        else:
            return 0.5,0.5

In [110]:
# Creating starter ELOs, initally assumming a flat 1500 at beginning

STARTING_ELO = 1500
MINIMUM_GAMES_PLAYED = 30

GAME_WEIGHTS = {
    'Friendly':20,
    'FIFA World Cup':60,
    'FIFA World Cup qualification':40,
    'Copa América':50,
    'African Cup of Nations':50,
    'UEFA Nations League':50,
    'CONCACAF Nations League':50,
    'UEFA Euro':50,
    'CONCACAF Championship':50
}

def filter_matches_for_elo(df,stop_date=None,minimum_games=MINIMUM_GAMES_PLAYED):

    filtered_by_date = df[df['date'] < stop_date] if stop_date is not None else df

    teams = generate_team_list(filtered_by_date,minimum_games)
    teams_mask = (filtered_by_date['home_team'].isin(teams)) & (filtered_by_date['away_team'].isin(teams)) 

    return filtered_by_date[teams_mask].reset_index(drop=True), teams


def elo_generation(matches_df,stop_date = None):


    matches_filtered, teams = filter_matches_for_elo(matches_df,stop_date) 

    current_rating = {team : STARTING_ELO for team in teams}
    history = []

    for row in matches_filtered.itertuples(index = True,name='match'):
        home_previous_rating = current_rating[row.home_team]
        away_previous_rating = current_rating[row.away_team]

        omega = 100 if not row.neutral else 0
        k0 = GAME_WEIGHTS.get(row.tournament,30)

        home_expected = elo_expected_outcome(home_previous_rating,away_previous_rating,omega = omega)
        away_expected = 1 - home_expected

        goal_difference = row.goal_diff

        home_outcome, away_outcome = result_convertion(row.result)

        new_home_rating = elo_rank_update(home_previous_rating,goal_difference,home_outcome,home_expected,k0=k0)
        new_away_rating = elo_rank_update(away_previous_rating,goal_difference,away_outcome,away_expected,k0=k0)
        
        current_rating[row.home_team] = new_home_rating
        current_rating[row.away_team] = new_away_rating

        history.append({'date':row.date, **current_rating})

    elo_df = pd.DataFrame(history).set_index('date')
    return elo_df[~elo_df.index.duplicated(keep='last')]



elo_df = elo_generation(played)


## Generating the full rating history

`elo_generation` walks through every match chronologically, starting every
team at a flat `STARTING_ELO` of 1500, and updates both teams' ratings after
each game using the formulas above. `GAME_WEIGHTS` encodes the relative
importance of each tournament type (World Cup highest, friendlies lowest;
30 is the default for any tournament not explicitly listed), and
`MINIMUM_GAMES_PLAYED` excludes teams with too short a history from the
rating pool entirely, so they can't appear as an opponent with an unreliable
rating. The result, `elo_df`, is a full Elo time series — one row per match
date, one column per team.

In [111]:
pd.set_option("display.max_rows", None)


top_thirty_teams = sorted(list(elo_df.iloc[-1].sort_values(ascending = False).head(10).index))

elo_df[wc_teams].iloc[-8:].sort_values(by='date')

final_ratings = dict(elo_df[wc_teams].iloc[-1])

not_played['home_rating'] = not_played['home_team'].map(final_ratings)
not_played['away_rating'] = not_played['away_team'].map(final_ratings)

not_played['expected_outcome'] = np.where(not_played['neutral'],
                               elo_expected_outcome(not_played['home_rating'],not_played['away_rating'],omega=0),
                               elo_expected_outcome(not_played['home_rating'],not_played['away_rating']))


expected_conditions = [
    not_played['expected_outcome'] >= 0.65,
    not_played['expected_outcome'] <= 0.35
]

expected_results = ['H','A']
not_played['expected_result'] = np.select(expected_conditions,expected_results,'D')

not_played


,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral,home_rating,away_rating,expected_outcome,expected_result
49434,2026-06-19,Turkey,Paraguay,NaN,NaN,FIFA World Cup,Santa Clara,United States,True,2065.227176,1955.037386,0.653465,H
49435,2026-06-19,Brazil,Haiti,NaN,NaN,FIFA World Cup,Philadelphia,United States,True,2242.078266,1753.536852,0.943335,H
49436,2026-06-19,Scotland,Morocco,NaN,NaN,FIFA World Cup,Foxborough,United States,True,2040.972417,2214.774783,0.268846,A
49437,2026-06-19,United States,Australia,NaN,NaN,FIFA World Cup,Seattle,United States,False,2086.336703,2164.418031,0.459675,D
49438,2026-06-20,Ecuador,Curaçao,NaN,NaN,FIFA World Cup,Kansas City,United States,True,2133.753657,1652.656979,0.941001,H
49439,2026-06-20,Netherlands,Sweden,NaN,NaN,FIFA World Cup,Houston,United States,True,2191.883257,2014.214696,0.735506,H
49440,2026-06-20,Tunisia,Japan,NaN,NaN,FIFA World Cup,Guadalupe,Mexico,True,1690.906833,2184.694465,0.055072,A
49441,2026-06-20,Germany,Ivory Coast,NaN,NaN,FIFA World Cup,Toronto,Canada,True,2205.256231,2072.950983,0.681703,H
49442,2026-06-21,Belgium,Iran,NaN,NaN,FIFA World Cup,Inglewood,United States,True,2129.846478,1979.578561,0.703707,H
49443,2026-06-21,New Zealand,Egypt,NaN,NaN,FIFA World Cup,Vancouver,Canada,True,1750.096232,1949.771397,0.240595,A


## Sanity-checking the ratings on unplayed 2026 World Cup fixtures

As a sense check before this rating becomes a model feature, each team's
most recent Elo is applied to every unplayed 2026 World Cup fixture to get a
win probability, then bucketed into an expected `H`/`D`/`A` result (using
0.65/0.35 thresholds — a deliberately wide "draw" band, since match outcome
in football is far noisier than a two-rating gap alone implies). This is
purely a qualitative check that the ratings produce sensible, well-calibrated
favourites; it isn't a model in itself.

**Next step (not yet built):** use this Elo time series — alongside match
context like home/away/neutral and tournament — as input features to a
Random Forest classifier trained on historical match outcomes, then use that
model in place of the fixed 0.65/0.35 thresholds above.